# 🧬 Python para Ciência de Dados: Saúde e Performance
**Objetivo da Aula:** Ingestão, Agregação e Cruzamento de múltiplas fontes de dados (Atividade, Sono, Nutrição, Medidas Corporais e Sinais Vitais) para extração holística de Insights de saúde.

## 1. Configuração do Ambiente e Ingestão de Dados
Vamos carregar a base de dados do nosso estudo de caso. Agora temos um pool de **5 arquivos separados**, o que simula perfeitamente o ecossistema fragmentado de aplicativos de saúde que usamos hoje em dia.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from functools import reduce

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")

# Lendo os cinco arquivos CSV
# NOTA: Certifique-se de ter feito o upload dos 5 arquivos no Colab
df_act = pd.read_csv('Activity.csv')
df_sleep = pd.read_csv('Sleep.csv')
df_nutri = pd.read_csv('Nutrition.csv')
df_body = pd.read_csv('Body_Measurements.csv')
df_vitals = pd.read_csv('Vitals.csv')

print("As 5 Bases de Dados foram carregadas com sucesso!")

## 2. Limpeza e Agregação de Dados (Groupby)
Precisamos garantir que todas as tabelas tenham uma coluna `Date` (Data) no mesmo formato para servir de "chave primária" e garantir **1 linha por dia**.

* **Truque de Limpeza:** O arquivo `Body_Measurements` possui a coluna `Date/Time` (com horas). Precisamos extrair apenas a data.
* **Regra de Agregação:** Para atividade física, usaremos a soma (`sum`) dos **Minutos de Atividade**. Para as demais métricas diárias como peso, usaremos a média (`mean`), e calorias o valor máximo do dia (`max`).

In [ ]:
# 2.1 Tratamento Especial: Extraindo a data da coluna Date/Time
df_body['Date'] = df_body['Date/Time'].str.split(' ').str[0]

# 2.2 Agregando as 5 fontes separadamente
act_agrupado = df_act.groupby('Date').agg({'Duration (min)': 'sum', 'Total Calories (kcal)': 'max'}).reset_index()

# Agregando o Sono (Vamos somar os minutos das fases do sono para ter o Total de Sono do dia)
sleep_agrupado = df_sleep.groupby('Date').agg({'Deep Sleep (min)': 'sum', 'Light Sleep (min)': 'sum', 'REM Sleep (min)': 'sum'}).reset_index()

nutri_agrupado = df_nutri.groupby('Date').agg({'Nutrition calories (kcal)': 'max', 'Protein (g)': 'max'}).reset_index()

body_agrupado = df_body.groupby('Date').agg({'Weight (kg)': 'mean', 'Body Fat (%)': 'mean'}).reset_index()

vitals_agrupado = df_vitals.groupby('Date').agg({'Resting heart rate avg (bpm)': 'mean'}).reset_index()

display(body_agrupado)

## 3. Cruzamento de Dados (Merge) e Dicionário de Dados
Vamos unir as informações e aplicar uma etapa crucial da Ciência de Dados: a **Tradução das Variáveis**. Nomes técnicos dificultam a leitura; vamos renomeá-los para o nosso contexto.

In [ ]:
# Colocando todos os DataFrames numa lista para juntar de uma vez
lista_dfs = [act_agrupado, sleep_agrupado, nutri_agrupado, body_agrupado, vitals_agrupado]

# Juntando tudo pela coluna 'Date'
df_final = reduce(lambda left, right: pd.merge(left, right, on='Date', how='outer'), lista_dfs)

# Ordenando cronologicamente
df_final = df_final.sort_values('Date').reset_index(drop=True)

# Preenchimento Inteligente (Foward Fill para o Peso entre os marcos)
df_final['Weight (kg)'] = df_final['Weight (kg)'].ffill().bfill()
df_final['Body Fat (%)'] = df_final['Body Fat (%)'].ffill().bfill()

# Preenchendo buracos residuais de dias sem monitoramento
df_final = df_final.fillna(df_final.mean(numeric_only=True))

# Calculando o Sono Total em Horas
df_final['Sono Total (horas)'] = (df_final['Deep Sleep (min)'] + df_final['Light Sleep (min)'] + df_final['REM Sleep (min)']) / 60

# ------------------------------------------------------------
# TRADUÇÃO DE VARIÁVEIS (Facilitando a leitura da Matriz)
# ------------------------------------------------------------
df_final = df_final.rename(columns={
    'Duration (min)': 'Tempo Atividade (min)',
    'Total Calories (kcal)': 'Gasto Calórico (kcal)',
    'Nutrition calories (kcal)': 'Calorias Ingeridas (kcal)',
    'Protein (g)': 'Proteína (g)',
    'Resting heart rate avg (bpm)': 'BPM Repouso'
})

print(f"Tabela Final Consolidada com {len(df_final)} dias avaliáveis.")
display(df_final[['Date', 'Tempo Atividade (min)', 'Calorias Ingeridas (kcal)', 'Sono Total (horas)']].head(5))

## 4. Análise Exploratória: A Matriz de Correlação Focada
Agora que os nomes estão amigáveis, vamos entender a dinâmica rápida do nosso corpo: **Esforço Físico, Nutrição e Recuperação**.

In [ ]:
plt.figure(figsize=(10, 6))

# Selecionando apenas as colunas vitais para a nossa análise didática
colunas_para_correlacao = [
    'Tempo Atividade (min)', 
    'Gasto Calórico (kcal)',
    'Calorias Ingeridas (kcal)', 
    'Proteína (g)',
    'Sono Total (horas)', 
    'BPM Repouso'
]

# Calculando a correlação
matriz_corr = df_final[colunas_para_correlacao].corr()

# Plotando o Mapa de Calor (Heatmap)
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, vmin=-1, vmax=1)

plt.title('Correlação Dinâmica: Esforço, Nutrição e Recuperação', fontsize=15)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Insights Visuais Focados
Vamos explorar três hipóteses biológicas provadas com dados.

### Insight A: O Impacto do Tempo de Atividade no Sono Total
O gráfico de dispersão com muitas variáveis estava ruidoso. Como Cientistas de Dados, precisamos simplificar a comunicação.
Vamos usar o `pd.cut` para categorizar os dias em **Descanso, Moderado e Intenso** e analisar a distribuição do sono num **Boxplot**.

In [ ]:
plt.figure(figsize=(10, 6))

# 1. Engenharia de Atributos: Categorizando o Volume de Treino
# Criamos faixas de tempo baseadas numa aproximação da rotina (ex: menos de 30m, até 60m, mais de 60m)
df_final['Categoria de Treino'] = pd.cut(
    df_final['Tempo Atividade (min)'],
    bins=[-1, 30, 60, df_final['Tempo Atividade (min)'].max() + 1],
    labels=['Descanso / Leve (< 30m)', 'Moderado (30m - 60m)', 'Intenso (> 60m)']
)

# 2. Plotando o Boxplot para ver a estatística (mediana e quartis)
sns.boxplot(
    data=df_final,
    x='Categoria de Treino',
    y='Sono Total (horas)',
    palette='Set2',
    showfliers=False # Oculta outliers puros para o swarmplot mostrar
)

# 3. Adicionando os pontos reais (Swarmplot) por cima para mostrar a densidade dos dados
sns.swarmplot(
    data=df_final,
    x='Categoria de Treino',
    y='Sono Total (horas)',
    color=".2",
    size=6,
    alpha=0.7
)

plt.title('Distribuição do Tempo de Sono por Intensidade do Dia', fontsize=15)
plt.xlabel('Volume de Atividade Física Diária', fontsize=12)
plt.ylabel('Sono Total (Horas)', fontsize=12)
plt.tight_layout()
plt.show()

### Insight B: Frequência Cardíaca de Repouso vs Horas de Sono
Quando dormimos mais, o nosso coração amanhece mais calmo? Vamos validar visualmente com uma **Análise de Regressão Linear Simples**.

In [ ]:
plt.figure(figsize=(9, 5))

# Gráfico de Regressão
sns.regplot(
    data=df_final,
    x='Sono Total (horas)',
    y='BPM Repouso',
    scatter_kws={'s': 100, 'alpha': 0.6, 'color': 'teal'},
    line_kws={'color': 'darkred', 'lw': 2}
)

plt.title('Tendência de Recuperação: Horas de Sono vs Freq. Cardíaca de Repouso', fontsize=14)
plt.xlabel('Sono Total (Horas)', fontsize=12)
plt.ylabel('Frequência Cardíaca de Repouso (BPM)', fontsize=12)
plt.tight_layout()
plt.show()

### Insight C: O Efeito do Refluxo (Alimentação vs Sono)
Uma queixa comum de saúde é o refluxo noturno. Existe uma crença empírica de que "comer demais afeta o sono". Vamos ver se os dados refletem essa biologia, plotando as Calorias Ingeridas contra as Horas de Sono.

In [ ]:
plt.figure(figsize=(9, 5))

# Gráfico de Regressão evidenciando o padrão de queda do sono
sns.regplot(
    data=df_final,
    x='Calorias Ingeridas (kcal)',
    y='Sono Total (horas)',
    scatter_kws={'s': 100, 'alpha': 0.6, 'color': 'darkorange'},
    line_kws={'color': 'black', 'lw': 2, 'linestyle': '--'}
)

plt.title('O Paradoxo da Nutrição: Calorias Ingeridas vs Tempo de Sono', fontsize=14)
plt.xlabel('Total de Calorias Ingeridas no Dia (kcal)', fontsize=12)
plt.ylabel('Sono Total (Horas)', fontsize=12)
plt.tight_layout()
plt.show()